# 연월별 가격정보 월간 자동 수집 (Monthly Job)

매월 1일과 말일에 스케줄러(Databricks Jobs)에 의해 실행되며, **전월의 연월별 가격정보**를 공공데이터 API에서 호출하여 저장합니다.

### 구조
- `api_client.py`의 `ATAPIClient.fetch_month_price()`를 사용합니다.
- `ref_sheet_시군구코드.csv`를 참조하여 전체 지역을 순회합니다.
- 지역별 페이지네이션으로 모든 데이터를 수집합니다.
- 저장 경로: `/Volumes/data_api/agrofood_permonth/test/연월별가격정보_YYYYMM.csv`

In [0]:
# api_client 즉시 반영되도록 설정
%load_ext autoreload
%autoreload 2

In [0]:
import json
import time
import os
import pandas as pd
from datetime import datetime, timedelta
from api_client import ATAPIClient

# API 키 설정
at_key = dbutils.secrets.get(scope="agrofood-api-prod", key="publicdata-service-key_yjs")

# 클라이언트 초기화
at_client = ATAPIClient(at_key)

### 1. 대상 연월 설정
- 기본값은 **전월(지난달)** 입니다.
- 매월 1일 또는 말일에 실행하면 자동으로 전월을 대상으로 수집합니다.
- 특정 연월을 수집하려면 `target_month`를 직접 변경하세요.

In [0]:
# 연월 설정 (Databricks widget 또는 수동 설정)

# 전월 자동 계산 (기본값)
today = datetime.now()
first_of_this_month = today.replace(day=1)
last_month = first_of_this_month - timedelta(days=1)
target_month = last_month.strftime("%Y%m")

# 수동 지정 (테스트용, 필요시 주석 해제)
#target_month = "202603"

print(f"[INFO] 수집 대상 연월: {target_month}")

### 2. 시군구코드 참조 데이터 불러오기
- `ref_sheet_시군구코드.csv` 파일을 읽어 지역 리스트를 생성합니다.
- 연월별 가격정보 API는 `sgg_cd`(시군구코드)가 필수 파라미터입니다.

In [0]:
# 시군구코드 참조 파일 경로
ref_file_path = '/Workspace/Users/biod1614@gmail.com/ref_sheet_시군구코드.csv'

try:
    df_sgg = pd.read_csv(ref_file_path, encoding='utf-8-sig')
except:
    df_sgg = pd.read_csv(ref_file_path, encoding='cp949')

print(f"[INFO] 시군구코드 참조 데이터: {len(df_sgg)}개 지역 로드 완료")
display(df_sgg)

### 3. 연월별 가격정보 수집
- `at_client.fetch_month_price()`를 사용하여 전체 지역의 전월 가격 데이터를 수집합니다.
- 지역별로 페이지네이션 루프를 돌며 전체 데이터를 가져옵니다.

In [0]:
print(f"{'='*60}")
print(f" 연월별 가격정보 수집 시작")
print(f" 대상 연월: {target_month}")
print(f" 수집 시작 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*60}")

# 출력 디렉토리 설정 (Databricks Volumes)
output_dir = "/Volumes/bronze_api/agrofood_peryearmonth/volumn"
os.makedirs(output_dir, exist_ok=True)

all_collected_items = []
success_count = 0
no_data_count = 0

for index, row in df_sgg.iterrows():
    sgg_cd = str(row['시군구코드'])
    sgg_nm = row['시군구명']
    
    print(f"\n[{index+1}/{len(df_sgg)}] {sgg_nm} (코드:{sgg_cd}) 수집 중...")
    
    sgg_items = []
    page = 1
    rows_per_page = 1000
    is_success = True
    
    while True:
        # fetch_month_price의 파라미터는 (YYYYMM, YYYYMM, sgg_cd) 형식이어야 함
        month_data = at_client.fetch_month_price(
            month_gte=target_month,
            month_lte=target_month,
            sgg_cd=sgg_cd,
            page=page,
            rows=rows_per_page,
            return_type="json"
        )
        
        # API 에러 처리
        if month_data is None:
            print(f" -> API 통신 에러 발생. 재시도...")
            time.sleep(2)
            month_data = at_client.fetch_month_price(
                month_gte=target_month,
                month_lte=target_month,
                sgg_cd=sgg_cd,
                page=page,
                rows=rows_per_page,
                return_type="json"
            )
            if month_data is None:
                print(f" -> 재시도 실패. 이 지역을 건너뜁니다.")
                is_success = False
                break
        
        body = month_data.get('response', {}).get('body', {})
        items = body.get('items', {}).get('item', [])
        total_count = body.get('totalCount', 0)
        
        if isinstance(items, dict):
            items = [items]
        
        if not items:
            break
        
        sgg_items.extend(items)
        
        if len(sgg_items) >= int(total_count):
            break
        
        page += 1
        time.sleep(0.05)  # API 부하 방지
    
    if sgg_items:
        all_collected_items.extend(sgg_items)
        success_count += 1
        print(f" -> {sgg_nm}: {len(sgg_items)}건 수집 완료")
    else:
        no_data_count += 1
        print(f" -> {sgg_nm}: 해당 연월 데이터 없음")

print(f"\n{'='*60}")
print(f" 수집 완료 요약")
print(f" 대상 연월: {target_month}")
print(f" 데이터 있는 지역: {success_count}개")
print(f" 데이터 없는 지역: {no_data_count}개")
print(f" 총 수집 건수: {len(all_collected_items)}건")
print(f" 완료 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*60}")

### 4. 수집 결과 저장
- `target_month` 기준으로 한 달치 CSV 파일 하나를 저장합니다.
- 저장 경로: `/Volumes/data_api/agrofood_permonth/test/연월별가격정보_YYYYMM.csv`

In [0]:
import pytz
kst = pytz.timezone('Asia/Seoul')

if all_collected_items:
    df_result = pd.DataFrame(all_collected_items)
    
    df_result['collect_time'] = datetime.now(kst).strftime('%Y-%m-%d_%H:%M:%S')
    
    save_path = os.path.join(output_dir, f"연월별가격정보_{target_month}.csv")
    df_result.to_csv(save_path, mode='w', encoding='utf-8', index=False)
    
    print(f"총 {len(all_collected_items)}건 CSV 저장 완료!")
    print(f"저장 경로: {save_path}")
    
    # 결과 미리보기
    print(f"\n수집된 연월별 가격 데이터 미리보기 (총 {len(df_result)}건):")
    display(df_result.head(10))

else:
    print(f"[INFO] {target_month} 연월에 수집된 데이터가 없습니다.")
